# **Exploration**

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ==============================
# 1. CHARGEMENT DONNEES
# ==============================
df = pd.read_parquet("data/votes.parquet")
print("Shape du dataset :", df.shape)
print("\nColonnes disponibles :")
print(df.columns.tolist())

# ==============================
# 2. TYPES ET VALEURS MANQUANTES
# ==============================
print("\nTypes des colonnes :")
print(df.dtypes)

print("\nValeurs manquantes par colonne :")
print(df.isna().sum())

# ==============================
# 3. STATISTIQUES DESCRIPTIVES
# ==============================
print("\nStatistiques numériques :")
print(df.describe())

print("\nStatistiques catégorielles (top 10 valeurs par colonne) :")
for col in ["model_a_name","model_b_name","chosen_model_name","model_pos","selected_category"]:
    if col in df.columns:
        print(f"\nColonne : {col}")
        print(df[col].value_counts().head(10))


Shape du dataset : (157132, 34)

Colonnes disponibles :
['id', 'timestamp', 'model_a_name', 'model_b_name', 'model_pair_name', 'chosen_model_name', 'opening_msg', 'both_equal', 'conversation_a', 'conversation_b', 'conv_turns', 'selected_category', 'is_unedited_prompt', 'conversation_pair_id', 'session_hash', 'visitor_id', 'conv_comments_a', 'conv_comments_b', 'conv_useful_a', 'conv_useful_b', 'conv_creative_a', 'conv_creative_b', 'conv_clear_formatting_a', 'conv_clear_formatting_b', 'conv_incorrect_a', 'conv_incorrect_b', 'conv_superficial_a', 'conv_superficial_b', 'conv_instructions_not_followed_a', 'conv_instructions_not_followed_b', 'system_prompt_b', 'system_prompt_a', 'conv_complete_a', 'conv_complete_b']

Types des colonnes :
id                                           int64
timestamp                           datetime64[us]
model_a_name                                object
model_b_name                                object
model_pair_name                             object
cho

# **Quantification du biais de longueur**

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import pearsonr
import statsmodels.api as sm
from sklearn.preprocessing import LabelEncoder
import scipy.sparse as sp

# ==============================
# 1. CHARGEMENT DONNEES
# ==============================
df = pd.read_parquet("data/votes.parquet")
df["total_conv_a_output_tokens"] = df["conversation_a"].apply(lambda x: len(str(x).split()))
df["total_conv_b_output_tokens"] = df["conversation_b"].apply(lambda x: len(str(x).split()))
cols_needed = ["model_a_name","model_b_name","chosen_model_name",
               "total_conv_a_output_tokens","total_conv_b_output_tokens"]
assert all(c in df.columns for c in cols_needed), "Colonnes manquantes !"

# ==============================
# 2. DIFFÉRENCE DE LONGUEUR
# ==============================
df["delta_tokens"] = df["total_conv_a_output_tokens"] - df["total_conv_b_output_tokens"]

df["vote_a"] = (df["chosen_model_name"] == df["model_a_name"]).astype(int)

# ==============================
# 3. CORRÉLATION SIMPLE
# ==============================
corr, pval = pearsonr(df["delta_tokens"], df["vote_a"])
print("Corrélation delta_tokens vs vote A :", corr)
print("p-value :", pval)

if corr > 0:
    print("Les réponses plus longues pour A tendent à être votées plus souvent.")
elif corr < 0:
    print("Les réponses plus longues pour B tendent à être votées plus souvent.")
else:
    print("Pas d’effet apparent de la longueur sur le vote.")

# ==============================
# 4. RÉGRESSION LOGISTIQUE AVEC LONGUEUR COMME COVARIABLE
# ==============================
X = sm.add_constant(df[["delta_tokens"]])  
y = df["vote_a"]

logit_model = sm.Logit(y, X)
result = logit_model.fit(disp=False)

print("\n=== Régression logistique ===")
print(result.summary())

# ==============================
# 5. CLASSEMENT BRADLEY-TERRY VIA LOGIT (AVEC ET SANS CORRECTION)
# ==============================

all_models = sorted(pd.concat([df["model_a_name"], df["model_b_name"]]).unique())
model_to_col = {m: i for i, m in enumerate(all_models)}

n_models = len(all_models)
n_votes  = len(df)

rows_a = df["model_a_name"].map(model_to_col).values
rows_b = df["model_b_name"].map(model_to_col).values
idx    = np.arange(n_votes)

X_model = sp.csr_matrix(
    (np.concatenate([np.ones(n_votes), -np.ones(n_votes)]),
     (np.concatenate([idx, idx]),
      np.concatenate([rows_a, rows_b]))),
    shape=(n_votes, n_models)
).toarray()

X_model_id = X_model[:, :-1]
model_names_id = all_models[:-1]
model_ref = all_models[-1]

# ------------------------------
# (A) BT SANS CORRECTION
# ------------------------------
X_bt = X_model_id
y = df["vote_a"].values

model_bt = sm.Logit(y, X_bt)
res_bt = model_bt.fit(disp=False)

coefs_bt = pd.Series(res_bt.params, index=model_names_id)

alphas_bt = np.append(coefs_bt.values, 0.0)
betas_bt = np.exp(alphas_bt)
betas_bt /= betas_bt.sum()

df_rank_bt = pd.DataFrame({
    "model": all_models,
    "beta": betas_bt
}).sort_values("beta", ascending=False).reset_index(drop=True)

df_rank_bt["rank"] = df_rank_bt.index + 1


# ------------------------------
# (B) BT AVEC CORRECTION
# ------------------------------
X_full = np.hstack([
    X_model_id,
    df[["delta_tokens"]].values
])

model_bt_corr = sm.Logit(y, X_full)
res_bt_corr = model_bt_corr.fit(disp=False)

coefs = pd.Series(
    res_bt_corr.params,
    index=[f"model__{m}" for m in model_names_id] + ["delta_tokens"]
)

print("\nEffet de la longueur (delta_tokens) :")
print(coefs["delta_tokens"])

model_coefs = coefs[[f"model__{m}" for m in model_names_id]]

alphas_corr = np.append(model_coefs.values, 0.0)
betas_corr = np.exp(alphas_corr)
betas_corr /= betas_corr.sum()

df_rank_corr = pd.DataFrame({
    "model": all_models,
    "beta_corr": betas_corr
}).sort_values("beta_corr", ascending=False).reset_index(drop=True)

df_rank_corr["rank_corr"] = df_rank_corr.index + 1


# ------------------------------
# 6. COMPARAISON DES CLASSEMENTS
# ------------------------------
df_compare = df_rank_bt.merge(df_rank_corr, on="model")

df_compare["rank_diff"] = df_compare["rank"] - df_compare["rank_corr"]

print("\nTop 10 AVANT correction :")
print(df_rank_bt.head(10))

print("\nTop 10 APRÈS correction :")
print(df_rank_corr.head(10))

print("\nModèles les plus impactés par la correction :")
print(df_compare.reindex(df_compare["rank_diff"].abs().sort_values(ascending=False).index).head(10))

Corrélation delta_tokens vs vote A : 0.049614490863686765
p-value : 3.25819917860322e-86
Les réponses plus longues pour A tendent à être votées plus souvent.

=== Régression logistique ===
                           Logit Regression Results                           
Dep. Variable:                 vote_a   No. Observations:               157132
Model:                          Logit   Df Residuals:                   157130
Method:                           MLE   Df Model:                            1
Date:                Wed, 25 Mar 2026   Pseudo R-squ.:                0.002055
Time:                        19:19:39   Log-Likelihood:            -1.0016e+05
converged:                       True   LL-Null:                   -1.0037e+05
Covariance Type:            nonrobust   LLR p-value:                 1.082e-91
                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------
const           -

c:\Users\Simon\.pyenv\pyenv-win\versions\3.11.7\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "



Effet de la longueur (delta_tokens) :
0.00018795352082005552

Top 10 AVANT correction :
                           model      beta  rank
0             mistral-medium-3.1  0.017812     1
1             mistral-large-2512  0.016837     2
2         gemini-3-flash-preview  0.016775     3
3               gemini-2.5-flash  0.016143     4
4            mistral-medium-2508  0.016083     5
5  gemini-3.1-flash-lite-preview  0.015815     6
6               gemini-2.0-flash  0.015297     7
7               magistral-medium  0.015152     8
8                        gpt-5.4  0.014545     9
9               deepseek-v3-0324  0.014095    10

Top 10 APRÈS correction :
                           model  beta_corr  rank_corr
0         gemini-3-flash-preview   0.016994          1
1             mistral-medium-3.1   0.016808          2
2  gemini-3.1-flash-lite-preview   0.016120          3
3             mistral-large-2512   0.015843          4
4               magistral-medium   0.015626          5
5            mi

c:\Users\Simon\.pyenv\pyenv-win\versions\3.11.7\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


### Corrélation entre longueur et vote
La corrélation positive indique que lorsque la réponse du modèle A est plus longue que la réponse du modèle B, elle est légèrement plus souvent choisi. La corrélation étant faible (env. 0.05), l'effet existe mais il est faible en pratique. La p-value étant extrêmement petite, l'effet est hautement significatif statistiquement.

### Régression logistique

#### Effet de la longueur (delta_tokens)
Coefficient : 0.0002
z-score : 19.7
p-value : env. 0

Plus A parle que B, plus A a de chances d'être choisi. L'effet est très petit par token. La longueur aide mais elle n'est clairement pas le facteur principal.

#### Constante (intercept)
Coefficient : -0.6805
Quand delta_tokens = 0, A est moins susceptible d'être choisi que B. Cela suggère un biais global (dataset, position, modèles).

### Classement de Bradley-Terry

On remarque que le top 10 des modèles change mais pas drastiquement. Le biais de longueur existe mais il ne renverse pas totalement le classement. On remarque que certains modèles chutent ce qui montre qu'ils étaient avantagés par des réponses longues tandis que d'autres montent, ils produisent sans doute des réponses plus courtes mais efficaces.

### Conclusion
Les réponses plus longues sont légèrement favorisées. Cet effet est statistiquement robuste mais faible en pratique.
On ne peut pas conclure que la longueur cause le vote ni que les modèles plus longs sont meilleurs.
On peut formuler l'hypothèse que la longueur est un gage de qualité, les meilleures réponses sont souvent plus longues.

# **Biais de position**

In [8]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency

# ==============================
# 1. CHARGEMENT DONNEES
# ==============================
df = pd.read_parquet("data/reactions.parquet")

cols_needed = ["model_pos","liked","creative","useful","incorrect"]
for c in cols_needed:
    if c not in df.columns:
        raise ValueError(f"Colonne {c} manquante dans le dataset")

# ==============================
# 2. FONCTION TEST CHI2 ET TAUX PAR POSITION
# ==============================
def test_position_effect(df, col):
    """
    df : dataframe
    col : variable binaire à tester (liked, creative, useful, incorrect)
    """
    table = pd.crosstab(df["model_pos"], df[col])
    chi2, p, dof, expected = chi2_contingency(table)
    
    rates = table.div(table.sum(axis=1), axis=0)[True] 
    
    print(f"\n--- Variable : {col} ---")
    print("Tableau de contingence :\n", table)
    print("Taux par position (proportion de True) :\n", rates)
    print(f"Chi2 = {chi2:.3f}, df = {dof}, p-value = {p:.3e}")
    
    if p < 0.05:
        print("→ Effet de position significatif")
    else:
        print("→ Pas d'effet significatif de position")

# ==============================
# 3. TEST POUR CHAQUE QUALIFICATIF
# ==============================
for var in ["liked","creative","useful","incorrect"]:
    df[var] = df[var].astype(bool)
    test_position_effect(df, var)


--- Variable : liked ---
Tableau de contingence :
 liked      False  True 
model_pos              
a          15213  31474
b          15075  32532
Taux par position (proportion de True) :
 model_pos
a    0.674149
b    0.683345
Name: True, dtype: float64
Chi2 = 9.100, df = 1, p-value = 2.556e-03
→ Effet de position significatif

--- Variable : creative ---
Tableau de contingence :
 creative   False  True 
model_pos              
a          43624   3063
b          44316   3291
Taux par position (proportion de True) :
 model_pos
a    0.065607
b    0.069128
Name: True, dtype: float64
Chi2 = 4.595, df = 1, p-value = 3.206e-02
→ Effet de position significatif

--- Variable : useful ---
Tableau de contingence :
 useful     False  True 
model_pos              
a          35911  10776
b          36007  11600
Taux par position (proportion de True) :
 model_pos
a    0.230814
b    0.243662
Name: True, dtype: float64
Chi2 = 21.427, df = 1, p-value = 3.676e-06
→ Effet de position significatif

--- 

### Variable liked
 Taux de réactions positives :
 - Position A : 67.41%
 - Position B : 68.33%
 Test statistique : X2 = 9.1 et p-value = 2.56e-03

 La différence est faible (+0.9 point pour B) mais statistiquement significative.

 ### Variable creative
Taux de créativité :
- Position A : 6.56%
- Position B : 6.91%
Test statistique : X2 = 4.6 et p-value = 0.032

La différence est très faible (+0.35 point pour B) mais elle est significative.

### Variable useful
Taux d'utilité :
- Position A : 23.08%
- Position B : 24.37%
Test statistique : X2 = 21.43 et p-value = 3.68e-06

La différence est un peu plus marquée mais elle reste faible (+1.3 point pour B), elle est très significative.

### Variable incorrect
Taux d'erreur :
- Position A : 10.96%
- Position B : 10.77%
Test statistique : X2 = 0.87 et p-value = 0.352

La différence est très faible et non significative.

### Conclusion

La réponse B est légèrement favorisé pour les caractéristiques liked, creative et useful. L'effet est faible mais significatif statistiquement.
Il n'y a pas d'effet sur la caractéristique incorrect.
En conclusion, il existe un léger biais de position en faveur de B.

# **Protocole d'annotation amélioré**

### Objectifs
Réduire simultanément les biais suivants tout en limitant l’augmentation du taux d’abandon des utilisateurs à ≤ 15 % :
*   Biais de position
*   Biais de longueur
*   Biais de sélection de prompts

### 1. Randomisation des positions
*   **Méthode :** Présentation aléatoire des réponses A/B pour chaque annotateur.
*   **Contrebalancement complet :** Chaque paire de modèles apparaît une fois en position A et une fois en position B.
*   **Justification :** Le biais de position est documenté dans les évaluations de dialogues (*Raffel et al., 2025; EMNLP 2025*).
*   **Impact attendu :** Réduction des préférences liées à la position sans altérer l’expérience utilisateur.

### 2. Équilibrage adaptatif par longueur
*   **Méthode :** Masquage ou tronquage des réponses au-delà d’une fenêtre maximale (ex. 400 tokens).
*   **Fonctionnalité :** Option « Voir tout » pour accéder à la réponse complète.
*   **Justification :** Les réponses longues sont perçues comme plus créatives/utiles (*Lengua et al., 2024; arXiv:2401.12491*).
*   **Impact attendu :** Diminution du biais de longueur tout en maintenant un taux d’abandon minimal.

### 3. Sélection contrainte des prompts
*   **Méthode :** Stratification par catégorie pour équilibrer la fréquence d’apparition.
    *   *Sur-représentées :* Fréquence réduite.
    *   *Sous-représentées :* Fréquence augmentée.
*   **Justification :** Le biais de sélection des prompts peut déformer les évaluations (*BYU et al., 2025; arXiv:2505.21116*).
*   **Impact attendu :** Couverture équilibrée des catégories et réduction de la sur-représentation de prompts créatifs.

### 4. Validation du protocole
Comparaison A/B interne (version standard vs. version améliorée) :
*   **Mesures :** Biais estimés (chi-2, régression), taux d’abandon, variance intra- et inter-annotateurs.
*   **Objectif :** Confirmer la réduction des biais tout en maintenant l’engagement utilisateur.